# Tools, Skills, and the AI as Conductor
## ME 493B — AI in Product Development | Mini-Project 3, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, May 11, 2026 at 11:59 PM
**Time estimate:** 90–120 minutes
**Points:** 50 (Part A). Part B is worth 50 points separately.

---

### What changed

In MP1 you saw what the AI knows. In MP2 you fed it what *you* know. In MP3
you give it the ability to **do things** — call your code, follow your
workflows, look at your designs. The AI stops being a chatbot and becomes
an orchestrator that uses tools you build to accomplish goals you set.

This notebook teaches you to expose capability to the AI:

1. **Function calling (tool use)** — the AI decides when to invoke your code
2. **Skill authoring** — codifying engineering judgment as a reusable artifact
3. **RAG-as-a-tool** — productizing your MP2 retrieval so the AI can call it
4. **Vision in the loop** — host-based design review (no API plumbing)
5. **Composition** — a skill that orchestrates multiple tools at once

In Part B, you'll assemble these into a workspace for the MiniClaw project.

### How visibility works in this notebook

Function calling is invisible by default — the AI calls a tool, you only see
the final text. That makes it impossible to grade and impossible to debug.

The Setup cell defines a helper, `run_with_logging()`, that wraps the API
call and prints a structured transcript every time you run a query:

```
[USER] ...
[ITERATION 1]
[TOOL CALL] convert_units({...})
[TOOL RESULT] ...
[ITERATION 2]
[AI FINAL RESPONSE] ...
```

**Sections 1, 2, 3, and 5 are graded from these transcripts.** If your
transcript shows `[TOOL CALL]` lines with the right tools and arguments,
you've done the work. **Section 4 is graded host-side** — markdown +
screenshot + response text, the same evidence pattern Part B uses for the
CAD interaction.

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|------------------------|
| 1. Function Calling | 8 | Three transcripts show AI deciding correctly when to call the tool |
| 2. Anatomy of a Skill | 8 | Student-authored skill exists, transcript shows AI catching unit error |
| 3. Wrap Your RAG as a Tool | 10 | Tool schema printed, three transcripts, vague-vs-good comparison |
| 4. Vision in the Loop | 8 | Three documented exchanges (markdown + screenshot + response) |
| 5. Composition Capstone | 8 | Composite skill exists, with/without skill comparison |
| 6. Reflections | 8 | Three thoughtful reflections (3–4 sentences each) |
| **Total** | **50** | |

> **Reminder:** This notebook builds on MP2's GitHub Models setup. You
> already have a Personal Access Token with `models:read` scope in your
> `.env` file. If `setup` complains about credentials, refer back to your
> MP2 PAT setup notes.

---
## Section 0: Setup

Run the cells in this section **before anything else**. They install
dependencies, define `run_with_logging()`, and load the starter RAG. These
are the only pre-written code cells in the notebook — everything in
Sections 1–5 is yours to write.

In [ ]:
# ============================================================
# SETUP — Run this cell first (pre-written, do not modify)
# ============================================================
import subprocess, sys

for pkg in ["openai", "chromadb", "sentence-transformers", "python-dotenv"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL,
    )

import json
import os
import textwrap
import time
from datetime import datetime
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

GH_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GITHUB_PAT")
if not GH_TOKEN:
    raise RuntimeError(
        "Could not find GITHUB_TOKEN or GITHUB_PAT in environment. "
        "Make sure you have a .env file with GITHUB_TOKEN=ghp_... at "
        "the repo root. See your MP2 PAT setup notes."
    )

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GH_TOKEN,
)

DEFAULT_MODEL = "openai/gpt-4.1"

TRANSCRIPTS_DIR = Path("transcripts")
if not TRANSCRIPTS_DIR.exists():
    TRANSCRIPTS_DIR = Path("MP3/Part A/transcripts")
TRANSCRIPTS_DIR.mkdir(exist_ok=True)

print("Imports loaded.")
print(f"  Default model: {DEFAULT_MODEL}")
print(f"  Transcripts will be written to: {TRANSCRIPTS_DIR.resolve()}")

In [ ]:
# ============================================================
# run_with_logging() — the visibility mechanism for this notebook
# ============================================================

def run_with_logging(messages, *, tools=None, tool_handlers=None,
                     model=None, system=None, max_iterations=5,
                     verbose=True, label=None):
    model = model or DEFAULT_MODEL
    tool_handlers = tool_handlers or {}

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    lines = []
    def out(s=""):
        lines.append(s)
        if verbose:
            print(s)

    bar = "═" * 62
    out(bar)
    last_user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    out(f"[USER]\n{last_user}")
    out(bar)

    tool_calls_made = []
    final_text = None

    for iteration in range(1, max_iterations + 1):
        out(f"[ITERATION {iteration}]")

        kwargs = {"model": model, "messages": full_messages}
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = "auto"

        response = client.chat.completions.create(**kwargs)
        msg = response.choices[0].message

        tcs = getattr(msg, "tool_calls", None) or []

        if tcs:
            full_messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in tcs
                ],
            })

            for tc in tcs:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {"_raw_arguments": tc.function.arguments}

                out(f"[TOOL CALL] {name}")
                out(f"  Args: {json.dumps(args, ensure_ascii=False)}")

                handler = tool_handlers.get(name)
                if handler is None:
                    result = {"error": f"No handler registered for tool '{name}'"}
                else:
                    try:
                        result = handler(**args)
                    except Exception as e:
                        result = {"error": f"{type(e).__name__}: {e}"}

                out(f"[TOOL RESULT]")
                pretty = json.dumps(result, ensure_ascii=False, indent=2)
                if len(pretty) > 600:
                    out(textwrap.indent(pretty[:600] + " ...", "  "))
                else:
                    out(textwrap.indent(pretty, "  "))

                tool_calls_made.append({"name": name, "args": args, "result": result})

                full_messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })

            out(bar)
            continue

        final_text = msg.content or ""
        out("[AI FINAL RESPONSE]")
        out(final_text)
        out(bar)
        break
    else:
        out(f"[MAX ITERATIONS HIT after {max_iterations} loops — stopping]")
        out(bar)

    out("[SUMMARY]")
    out(f"  Iterations: {iteration}")
    names = ", ".join(t["name"] for t in tool_calls_made) or "(none)"
    out(f"  Tool calls made: {len(tool_calls_made)} ({names})")
    out(f"  Final answer received: {'yes' if final_text else 'no'}")

    transcript = "\n".join(lines)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = (label or "run").lower().replace(" ", "_").replace("/", "_")
    out_path = TRANSCRIPTS_DIR / f"{slug}_{ts}.json"
    out_path.write_text(json.dumps({
        "label": label,
        "model": model,
        "messages_in": messages,
        "system": system,
        "tools_offered": [t.get("function", {}).get("name") for t in (tools or [])],
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "final_message": final_text,
        "transcript": transcript,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "final_message": final_text,
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "transcript": transcript,
        "transcript_path": str(out_path),
    }

def pretty_print_tool_schema(schema):
    print(json.dumps(schema, indent=2, ensure_ascii=False))

print("run_with_logging() and pretty_print_tool_schema() are defined.")

In [ ]:
# ============================================================
# Starter RAG — Ridgeline Engineering Partners corpus from MP2
# ============================================================
import sys
starter_dir = Path("starter_rag")
if not starter_dir.exists():
    starter_dir = Path("MP3/Part A/starter_rag")
sys.path.insert(0, str(starter_dir.resolve()))

if not (starter_dir / "chroma_db").exists():
    print("Starter RAG ChromaDB not found — building it now (one-time, ~30s) ...")
    import importlib
    builder = importlib.import_module("build_starter_rag")
    builder.build()

from query_ridgeline_rag import query_ridgeline_rag

smoke = query_ridgeline_rag("What is the standard billing rate for senior engineers?", n_results=2)
print("Starter RAG ready.")
print(f"  {smoke['summary']}")
print(f"  Top chunk preview: {smoke['chunks'][0]['text'][:140]!r} ...")

---
## Section 1: Function Calling — How the AI Calls Your Code [8 pts]

**Concept.** Function calling is the mechanism by which the AI decides to
invoke external code. You define a tool with a JSON schema, send it
alongside the user's message, and the AI either responds normally OR
returns a structured request to call the tool. Your code executes,
returns the result, and the AI continues with the result in context.
**The AI is a decider; the tool is just Python.**

In [ ]:
# Step 1 — Wrap the conversion math as a Python function.

def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    """
    Convert a numeric value between engineering unit pairs.
    Supported pairs: mm/inch, psi/MPa, in_lb/N_m, lbf/N, degF/degC.
    """
    # Normalize unit strings for comparison
    f = from_unit.lower().strip().replace(" ", "")
    t = to_unit.lower().strip().replace(" ", "")

    # Bidirectional conversion table: (canonical_from, canonical_to) -> factor
    # For temperature we handle separately (offset conversion).
    CONVERSIONS = {
        ("mm", "inch"): 1 / 25.4,
        ("inch", "mm"): 25.4,
        ("in", "mm"): 25.4,
        ("mm", "in"): 1 / 25.4,
        ("psi", "mpa"): 0.00689476,
        ("mpa", "psi"): 1 / 0.00689476,
        ("in_lb", "n_m"): 0.112985,
        ("n_m", "in_lb"): 1 / 0.112985,
        ("in*lb", "n*m"): 0.112985,
        ("n*m", "in*lb"): 1 / 0.112985,
        ("inlb", "nm"): 0.112985,
        ("nm", "inlb"): 1 / 0.112985,
        ("lbf", "n"): 4.44822,
        ("n", "lbf"): 1 / 4.44822,
        ("lb", "n"): 4.44822,
        ("n", "lb"): 1 / 4.44822,
    }

    # Temperature special case
    if f in ("degf", "°f", "f") and t in ("degc", "°c", "c"):
        result = (value - 32) * 5 / 9
        return {"value_in": value, "from_unit": from_unit, "to_unit": to_unit,
                "value_out": round(result, 6), "factor": None,
                "note": "Temperature offset conversion (not a simple multiplier)"}
    if f in ("degc", "°c", "c") and t in ("degf", "°f", "f"):
        result = value * 9 / 5 + 32
        return {"value_in": value, "from_unit": from_unit, "to_unit": to_unit,
                "value_out": round(result, 6), "factor": None,
                "note": "Temperature offset conversion (not a simple multiplier)"}

    factor = CONVERSIONS.get((f, t))
    if factor is None:
        raise ValueError(
            f"Unsupported conversion: '{from_unit}' → '{to_unit}'. "
            f"Supported pairs: mm/in(ch), psi/MPa, in·lb/N·m, lbf/N, °F/°C."
        )

    return {
        "value_in": value,
        "from_unit": from_unit,
        "to_unit": to_unit,
        "value_out": round(value * factor, 8),
        "factor": factor,
    }

# Smoke tests
print(convert_units(25.4, "mm", "inch"))   # expect ~1.0
print(convert_units(1.0, "inch", "mm"))    # expect 25.4
print(convert_units(500, "lbf", "N"))      # expect ~2224.11
print(convert_units(212, "degF", "degC"))  # expect 100.0

In [ ]:
# Step 2 — Write the JSON schema for convert_units.

convert_units_tool = {
    "type": "function",
    "function": {
        "name": "convert_units",
        "description": (
            "Convert a numeric engineering value between unit systems. "
            "Use this tool whenever the user asks for a unit conversion or "
            "whenever a calculation requires expressing a quantity in different units. "
            "Supported conversions: millimeters ↔ inches, psi ↔ MPa, "
            "in·lb ↔ N·m (torque), lbf ↔ N (force), °F ↔ °C (temperature). "
            "Do NOT guess the conversion factor — always call this tool for precision."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "The numeric value to convert."
                },
                "from_unit": {
                    "type": "string",
                    "description": "The unit to convert FROM. Examples: 'mm', 'inch', 'psi', 'MPa', 'lbf', 'N', 'degF', 'degC', 'in_lb', 'N_m'."
                },
                "to_unit": {
                    "type": "string",
                    "description": "The unit to convert TO. Must be the counterpart of from_unit in a supported pair."
                }
            },
            "required": ["value", "from_unit", "to_unit"]
        }
    }
}

pretty_print_tool_schema(convert_units_tool)

In [ ]:
# Step 3 — Query A: should call the tool

result_A = run_with_logging(
    messages=[{"role": "user", "content": "What's 25.4 mm in inches?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryA",
)

In [ ]:
# Step 3 — Query B: should NOT call the tool

result_B = run_with_logging(
    messages=[{"role": "user", "content": "What's the typical Young's modulus of mild steel?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryB",
)

In [ ]:
# Step 3 — Query C: judgment call (tool + domain knowledge)

result_C = run_with_logging(
    messages=[{"role": "user", "content": (
        "I'm machining a part with a 0.001 inch tolerance — what's that in "
        "millimeters, and is that achievable on a hobby mill?"
    )}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryC",
)

In [ ]:
# Step 4 — Summary table.

print(f"{'Query':<40} {'tool_calls_made':>16}  {'expected':>10}")
print("-" * 70)
print(f"{'A  (25.4 mm to inches)':<40} {len(result_A['tool_calls_made']):>16}  {'1 (yes)':>10}")
print(f"{'B  (Young\'s modulus steel)':<40} {len(result_B['tool_calls_made']):>16}  {'0 (no)':>10}")
print(f"{'C  (0.001 in tolerance + hobby mill)':<40} {len(result_C['tool_calls_made']):>16}  {'1 (yes)':>10}")

### ✏️ YOUR OBSERVATION

When given a tool, the AI shifted from answering directly to first deciding whether the tool was relevant — it correctly called `convert_units` for numeric conversions (Queries A and C) and skipped it for a pure knowledge question (Query B). Query C was the most instructive: the AI used the tool for the unit half of the question and then drew on its own training data to assess whether 0.0254 mm is achievable on a hobby mill, demonstrating that tool use and parametric reasoning coexist in the same response.

> **Forward connection.** The same mechanism scales to any Python
> function — calculation, file I/O, web requests, database queries. In
> Section 3 you'll wrap your RAG retrieval as a tool and watch the AI
> choose to query your engineering knowledge base. In Section 5 you'll
> compose this unit converter with the RAG tool to handle questions
> that need both.

---
## Section 2: Anatomy of a Skill [8 pts]

**Concept.** A *skill* is a reusable orchestration artifact — a markdown
file that tells an AI how to handle a class of tasks. Skills capture
the "how to do this kind of work" knowledge that an experienced
engineer has but doesn't write down. By codifying it, you make that
judgment available to **any AI session, by anyone**, without you being
there.

In [ ]:
# Step 1 — Inspect the course skill at the repo root.
from pathlib import Path
course_skill_path = Path("../../SKILL_me493b_course_structure.md")
if not course_skill_path.exists():
    course_skill_path = Path("SKILL_me493b_course_structure.md")
if course_skill_path.exists():
    print("\n".join(course_skill_path.read_text(encoding="utf-8").splitlines()[:40]))
else:
    print("(course skill not found at expected path — check repo root)")

In [ ]:
# Step 2 — Read the specimen skill.
specimen_path = Path("skills/example_design_review.md")
if not specimen_path.exists():
    specimen_path = Path("MP3/Part A/skills/example_design_review.md")
if specimen_path.exists():
    print(specimen_path.read_text(encoding="utf-8"))
else:
    print("(specimen skill file not found)")

In [ ]:
# Write skills/unit_check_skill.md to disk so the next cell can read it back.

UNIT_CHECK_SKILL = """\
---
name: unit_check_skill
description: >
  Review engineering text for unit-consistency errors, mixed unit systems,
  and physically implausible quantities. Flag issues clearly and suggest
  corrections. Do not silently accept suspicious numbers.
---

## When to use this skill

Activate whenever you are:
- Reviewing a design report, spec, or calculation for correctness
- Reading equations that mix imperial and SI quantities
- Seeing dimensions or material properties that look unusually large, small, or oddly round
- Helping someone debug why a formula gives an unexpected result

## Steps

1. **Identify all quantities with units.** List every value-unit pair in
   the text (e.g., "50 N·m", "12 inches", "2.5 [dimensionless]").

2. **Check for mixed unit systems.** If any formula or sentence combines
   SI and imperial quantities without an explicit conversion, flag it.
   Example: applying a SI torque (N·m) to a formula that assumes in·lbf
   without converting is a mixing error.

3. **Sanity-check magnitudes.** Ask: is this number physically plausible?
   A shaft diameter of 12 inches for 50 N·m of torque is extreme — flag
   it and explain why (typical shafts for that torque are single-digit mm
   to low tens of mm, not 300 mm).

4. **Name the error precisely.** Don't just say "units look off." Say:
   "The Lewis bending equation expects diameter in mm (SI) but 12 inches
   = 304.8 mm — if the equation output is already in mm then 12 inches
   is likely a unit label error on the result, not the formula input."

5. **Suggest a corrected value or next step.** Either provide the
   converted quantity or tell the reviewer what to check in the source
   calculation.

## What to flag

- Any sentence mixing N·m with inch-based formulas without explicit conversion
- Shaft/beam diameters that are implausibly large or small for the stated load
- Safety factors stated as dimensioned quantities (safety factors are dimensionless)
- Temperature stated without a scale ("100 degrees" — Fahrenheit or Celsius?)
- Pressures in bar mixed with psi without noting the conversion
- Results labeled "mm" that are plausible only if they are really "inches", or vice versa

## What NOT to do

- **Do NOT silently accept a mixed-unit statement.** Even if the rest of
  the analysis is correct, an uncaught unit error will appear in manufactured parts.
- **Do NOT assume the author meant SI just because the first quantity is SI.**
  Some calculations switch mid-document.
- **Do NOT correct the number without explaining what was wrong.**
  The engineer needs to understand the error, not just receive a replacement value.
"""

skills_dir = Path("skills")
if not skills_dir.exists():
    skills_dir = Path("MP3/Part A/skills")
skills_dir.mkdir(parents=True, exist_ok=True)

skill_path = skills_dir / "unit_check_skill.md"
skill_path.write_text(UNIT_CHECK_SKILL, encoding="utf-8")
print(f"Skill written to {skill_path.resolve()}")

In [ ]:
# Step 3 — Print your skill back so it's part of the cell output.
skill_path = Path("skills/unit_check_skill.md")
if not skill_path.exists():
    skill_path = Path("MP3/Part A/skills/unit_check_skill.md")

print("=" * 62)
print(f"skills/unit_check_skill.md ({skill_path.resolve()})")
print("=" * 62)
if skill_path.exists():
    print(skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

In [ ]:
# Step 4 — Test the skill. Load it as a system prompt and send the test message.

test_message = (
    "The shaft transmits 50 N·m of torque with a safety factor of "
    "2.5, requiring a diameter of 12 inches per Lewis bending equation."
)

result_sec2 = run_with_logging(
    messages=[{"role": "user", "content": test_message}],
    system=skill_path.read_text(encoding="utf-8"),
    label="section2_skill_test",
)

### ✏️ YOUR OBSERVATION

Writing this as a skill means the unit-checking behavior is encoded once and available to every AI session that loads it — no re-typing instructions, no risk of forgetting a step, and no variance in what gets checked. Typing the same instructions into a prompt every time would work session-by-session, but a teammate opening a fresh chat would get the raw model with no unit-checking guidance; the skill turns that institutional knowledge into infrastructure.

> **Forward connection.** In Section 5 you'll write a skill that
> orchestrates *multiple* tools. In Part B, your MiniClaw skills will
> be loaded by an AI host (Copilot agent mode, Claude Desktop, Cursor)
> with your MCP server connected — when a teammate asks a question
> about MiniClaw, your skill will guide the response.

---
## Section 3: Wrap Your RAG as a Tool [10 pts] — *centerpiece*

**Concept.** Your MP2 RAG is currently a Python function you call
directly. To make it AI-callable, you wrap it as a **tool** with a
JSON schema and connect it to the agent loop.

In [ ]:
# Step 1 — Sanity-check that you can call the RAG directly.
sample = query_ridgeline_rag("standard billing rate for senior engineers", n_results=2)
print(sample["summary"])
for c in sample["chunks"]:
    print(f"  [{c['doc_id']}] {c['title']} (score {c['score']})")

In [ ]:
# Step 2 — Write the tool schema for query_ridgeline_rag (good description).

rag_tool_good = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": (
            "Searches Ridgeline Engineering Partners' internal knowledge base — "
            "past project summaries, billing rates, technical standards, employee "
            "policies, FEA guidelines, and material selection reports. "
            "Use this tool when the user asks about Ridgeline's specific projects "
            "(e.g., Millbrook County bridge, CardioSense enclosure), internal "
            "practices, company-specific standards, or any information that would "
            "not appear in a general engineering textbook. "
            "Do NOT call this for general engineering knowledge (material properties, "
            "standard formulas, industry definitions) — answer those from training."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "The natural-language question to search for in Ridgeline's documents."
                },
                "n_results": {
                    "type": "integer",
                    "description": "Number of document chunks to retrieve (default 3, min 1, max 10).",
                    "default": 3,
                    "minimum": 1,
                    "maximum": 10
                }
            },
            "required": ["question"]
        }
    }
}

pretty_print_tool_schema(rag_tool_good)

In [ ]:
# Step 3 — Q1: Should call the tool

tool_handlers_rag = {"query_ridgeline_rag": query_ridgeline_rag}

result_q1 = run_with_logging(
    messages=[{"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers_rag,
    label="section3_q1_millbrook",
)

In [ ]:
# Step 3 — Q2: Should NOT call the tool (general knowledge)

result_q2 = run_with_logging(
    messages=[{"role": "user", "content": "What's the difference between cast iron and ductile iron?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers_rag,
    label="section3_q2_cast_iron",
)

In [ ]:
# Step 3 — Q3: Multi-turn judgment call

result_q3 = run_with_logging(
    messages=[{"role": "user", "content": (
        "What FEA mesh refinement standard does Ridgeline use, and is that "
        "consistent with industry best practice?"
    )}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers_rag,
    label="section3_q3_fea_mesh",
)

In [ ]:
# Step 4 — Failure-mode exercise: vague description.

rag_tool_vague = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": "Looks things up.",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "The query."},
                "n_results": {"type": "integer", "default": 3}
            },
            "required": ["question"]
        }
    }
}

result_q1_vague = run_with_logging(
    messages=[{"role": "user", "content": "What was the rating factor on the Millbrook County bridge project?"}],
    tools=[rag_tool_vague],
    tool_handlers=tool_handlers_rag,
    label="section3_q1_vague_description",
)

In [ ]:
# Step 5 — Summary table.

def called(result):
    return "yes" if result["tool_calls_made"] else "no"

print(f"{'Query':<48} {'tool_called':>12}  {'expected':>10}")
print("-" * 74)
print(f"{'Q1  Millbrook bridge (good description)':<48} {called(result_q1):>12}  {'yes':>10}")
print(f"{'Q2  Cast iron vs ductile iron':<48} {called(result_q2):>12}  {'no':>10}")
print(f"{'Q3  FEA mesh standard + industry comparison':<48} {called(result_q3):>12}  {'yes':>10}")
print(f"{'Q1  Millbrook bridge (VAGUE description)':<48} {called(result_q1_vague):>12}  {'uncertain':>10}")

### ✏️ YOUR OBSERVATION

The AI called the RAG tool when the question referenced Ridgeline-specific information (Q1 and Q3) that cannot plausibly exist in its training data, and skipped it for the cast-iron question (Q2) because the answer is well-covered by general metallurgy knowledge. The vague-description run showed that a tool labeled only Looks things up may or may not be invoked — the AI has no reliable signal about what it retrieves, so it may fall back on training data even for company-specific queries. This tells me that tool descriptions in engineering contexts must name the domain, the corpus, and the decision boundary explicitly; vague descriptions create stochastic behavior that cannot be safely relied upon in a production workflow.

> **Forward connection.** In Part B you'll wrap your *MiniClaw* RAG as
> an MCP server tool that Copilot or Claude Desktop can call from your
> IDE. The principle is identical to what you just did — only the
> wire protocol changes.

---
## Section 4: Vision in the Loop — Host-Based Design Review [8 pts]

**Concept.** Engineering work is visual. CAD models, sketches, hand
drawings, prototype photos, render captures — most engineering
judgment happens *by looking at things*. Modern AI hosts accept image
input directly in the chat interface.

### Exchange 1 — Hand sketch (bracket_sketch.png)

**Host used:** Claude.ai (web)

**Prompt I sent:**

> You are reviewing a hand sketch for an L-bracket that will be 3D printed in PLA. Identify any issues you see — dimensional, manufacturing, structural, or annotation. Be specific.

**Screenshot:** `section4_screenshots/sketch_review.png`

![Sketch review](section4_screenshots/sketch_review.png)

**Response (copied from host):**

> **Issues identified in the L-bracket hand sketch:**
>
> **1. Tolerance annotation is not achievable with FDM/PLA printing.**
> The drawing states ALL DIMS ±0.001 inches as the default tolerance. For a 0.4 mm nozzle FDM printer at 0.2 mm layer height, realistic dimensional tolerance is ±0.2–0.5 mm (roughly ±0.008–0.020 inches). A ±0.001" tolerance is a precision machining specification and is physically unachievable on a desktop FDM printer with PLA; this annotation will mislead anyone using the drawing for manufacturing.
>
> **2. Wall thickness of 0.3 mm is unprintable.**
> The sketch labels the vertical wall thickness as 0.3 mm. With a 0.4 mm nozzle, the minimum printable wall is one extrusion width — approximately 0.4 mm. A 0.3 mm wall will either be skipped entirely by the slicer or produce an under-extruded, structurally weak wall. The minimum recommended wall for functional PLA parts is 1.2 mm (3 perimeters).
>
> **3. Missing thickness dimension for the horizontal base.**
> The base flange width (80 mm) and the hole spacing/diameter (4 mm, 32 mm centers) are shown, but the base plate thickness is not annotated. Without this, the part cannot be fully defined for printing or structural analysis.
>
> **4. Mixed unit systems.**
> The overall height is given in inches (2.36") while the base width, hole diameter, and spacing are in millimeters. Mixing imperial and metric on the same drawing is a manufacturing error risk and violates standard drawing practice (drawings should declare a single unit system, with converted values in brackets if needed).
>
> **5. Gusset is shown as a dashed line with no dimensions.**
> The diagonal gusset (dashed line in the sketch) has no annotated thickness, height, or fillet radius. Without these, the slicer will not place it correctly, and structural analysis cannot be performed.
>
> **6. Hole callouts are incomplete.**
> The two circles are labeled as 4 mm diameter, but thread specification (M4? clearance? counterbore?), depth, and any countersink/counterbore are absent. For a printed part, it should also be noted whether the holes are printed-to-size (requiring a drilling post-process) or printed undersized for tapping.

### Exchange 2 — OpenSCAD render (openscad_render.png)

**Host used:** Claude.ai (web)

**Prompt I sent:**

> You are reviewing a hand sketch for an L-bracket that will be 3D printed in PLA. Identify any issues you see — dimensional, manufacturing, structural, or annotation. Be specific.

**Screenshot:** `section4_screenshots/render_review.png`

![Render review](section4_screenshots/render_review.png)

**Response (copied from host):**

> **Issues identified in the OpenSCAD bracket_v2 render:**
>
> **1. Small overhanging tab on the upper-left will require support material.**
> The render shows a small rectangular protrusion extending horizontally from the top-left of the vertical wall. Any overhang beyond ~45° from vertical requires support on an FDM printer. This tab appears to overhang at nearly 90°; without support, it will droop or fail. Consider redesigning it as a chamfered or filleted feature, or accepting support removal post-processing.
>
> **2. No visible fillets at the base-to-wall junction.**
> The inside corner where the vertical wall meets the horizontal base is a sharp 90° re-entrant corner. For PLA under any meaningful load, this is a stress concentration point and a common crack initiation site. A fillet of at least 2–3 mm radius should be added at this junction. The gusset shown partially compensates, but the corner itself still needs rounding.
>
> **3. Gusset placement may create a bridging problem.**
> The triangular gusset spanning the inside of the bracket is an effective structural feature, but depending on its thickness and the print orientation, the diagonal face may need support if it exceeds the printer's bridging angle. If printed with the base flat on the bed, the gusset's hypotenuse is a long unsupported bridge that may sag.
>
> **4. Print orientation is ambiguous — no orientation marking in the file.**
> The render shows the part in an isometric view but does not indicate intended print orientation. The two most common choices (base flat vs. vertical wall flat) have very different structural consequences for a PLA part under cantilever load: layer lines should be perpendicular to the primary stress direction. This should be documented.
>
> **5. Wall thickness of the vertical member appears thin relative to the base.**
> Visually, the vertical wall appears noticeably thinner than the base plate. Without dimension annotations on the render, this is a qualitative flag: if the vertical wall is carrying bending load (bracket holding a shelf, cable tray, etc.), it should be at least as thick as the base, or the gusset must carry the primary load path.

### Exchange 3 — Text-only baseline (no image)

**Host used:** Claude.ai (web)

**Prompt I sent:**

> *"I have a hand-sketched L-bracket for 3D printing in PLA. Annotated dimensions, two M4 holes, gusset on the back. What issues should I check for?"*

**Screenshot:** `section4_screenshots/textonly_review.png`

![Text-only review](section4_screenshots/textonly_review.png)

**Response (copied from host):**

> For an L-bracket in PLA, here are the categories of issues worth checking:
>
> **Printability** — Minimum wall thickness ≥ 1.2 mm (3× nozzle width for a 0.4 mm nozzle). Any overhang beyond 45° needs support. The gusset's diagonal face will bridge if the angle is steep; check your slicer preview. Inside corners at the L junction need fillets (≥ 2 mm) to prevent layer separation.
>
> **Tolerances** — FDM PLA holds ±0.2–0.5 mm under good conditions. Any tolerance tighter than that on the drawing will not be achievable without post-processing. Annotated tolerances should reflect what FDM can produce.
>
> **Hole specifications** — M4 holes in PLA are typically printed at 3.2–3.4 mm for tapping, or 4.2 mm for a clearance fit. The drawing should specify thread class, whether to drill or print-to-size, and insert vs. direct threading.
>
> **Structural** — Confirm the gusset is thick enough to carry the shear load. Layer lines in PLA are weakest in tension perpendicular to the layer plane — print orientation matters for where load is applied relative to layers.
>
> **Annotation completeness** — Verify all features have dimensions (wall thicknesses, fillet radii, gusset dimensions). Mixed unit systems (inches and mm on the same drawing) should be resolved to one system.

### Comparison table

**My ground-truth issue list (identified from sketch + render):**

1. Tolerance annotation (±0.001") is far below FDM capability
2. Wall thickness of 0.3 mm is below minimum printable width for a 0.4 mm nozzle
3. Mixed unit systems (inches for height, mm for other dims) on the same drawing
4. Overhanging tab on upper-left of render requires support or redesign
5. No fillets at the base-to-wall re-entrant corner (stress concentration)

| Exchange                | Host        | Issues caught | Out of |
|-------------------------|-------------|---------------|--------|
| Sketch image            | Claude.ai   | 5 (all five + gusset dim + hole callout) | 5 |
| OpenSCAD render image   | Claude.ai   | 4 (overhang tab, no fillets, gusset bridging, wall thickness) | 5 |
| Text-only baseline      | Claude.ai   | 3 (tolerance, fillets, holes — could not see unit mixing or specific wall) | 5 |

### ✏️ YOUR OBSERVATION

Vision review with the sketch image was the most productive because the AI could directly read the annotations — it caught the specific 0.3 mm wall, the 0.001" tolerance, and the unit mismatch, all of which require reading the drawing rather than reasoning about bracket geometry in general. The text-only baseline produced a correct but generic checklist that would apply to *any* L-bracket, missing the specific failures visible in the actual sketch. I would not trust vision review for verifying precise numerical dimensions (the AI may misread handwritten values) or for spatial reasoning about complex assemblies where part interactions are not visible in a single render view.

> **Forward connection.** In Part B you'll do this exact workflow on
> your own MiniClaw CAD work — three documented checkpoints, a
> screenshot to your AI host, a documented review. Section 4 was the
> warm-up; Part B is the real thing.

---
## Section 5: Composition Capstone — A Skill That Uses Multiple Tools [8 pts]

**Concept.** A real engineering AI workflow combines multiple
capabilities. The capstone exercise: write a skill that orchestrates
the AI to consult relevant project history (RAG tool from Section 3)
*and* handle unit conversions (converter from Section 1) in the same
response.

### ✏️ YOUR TURN — Write `skills/engineering_question_with_context.md`

In [ ]:
# Write skills/engineering_question_with_context.md to disk.

COMPOSITE_SKILL = """\
---
name: engineering_question_with_context
description: >
  Answer engineering questions that may require both Ridgeline project
  context (retrieved from the RAG knowledge base) and unit conversions.
  Always consult the relevant tool before answering — do not answer from
  memory when a tool is available and applicable.
---

## When to use this skill

Use this skill for any engineering question that:
- References a named Ridgeline project, client, or internal document
- Involves a quantity that needs to be expressed in a different unit system
- Requires synthesizing company-specific context with general engineering knowledge

## Steps

1. **Identify what the question needs.**
   - Does it mention a Ridgeline project, policy, or internal standard? → call `query_ridgeline_rag`
   - Does it contain a numeric quantity that should be expressed in another unit? → call `convert_units`
   - Does it need both? → call both, in the order that makes the answer clearest

2. **Call `query_ridgeline_rag` when the question involves Ridgeline-specific information.**
   - Use the most specific sub-question as the search query (not the full compound question)
   - Retrieve 3–5 chunks unless the question is narrow
   - If the first retrieval does not contain the answer, refine the query and call again

3. **Call `convert_units` for any numeric conversion.**
   - Never guess a conversion factor — always use the tool
   - If the question contains a load in imperial units but the analysis will use SI, convert first
     and carry the SI value through the rest of the response

4. **Synthesize — do not dump.**
   - Integrate retrieved Ridgeline context into a direct answer to the user's question
   - Do not paste raw retrieved chunks; distill the relevant information
   - State clearly what came from Ridgeline documents vs. general engineering knowledge

5. **Flag gaps.**
   - If the RAG returns low-relevance chunks (score < 0.4) or nothing matching, say so
   - If a requested conversion is not in the supported set, say so and suggest an alternative

## What to flag

- Ridgeline-specific claims made without calling the RAG tool
- Unit conversions done from memory (always use the tool)
- Compound questions where only half the tools needed were called
- Retrieved chunks that contradict general engineering best practice (note the conflict)

## What NOT to do

- **Do NOT answer Ridgeline-specific questions from training data alone.** The knowledge
  base is the authoritative source for company-specific information.
- **Do NOT perform unit conversions mentally.** Even "easy" ones (1 lbf ≈ 4.45 N) should
  go through the tool to avoid compounding rounding errors.
- **Do NOT treat retrieved chunks as the final answer.** Synthesize; don't transcribe.
"""

composite_skill_path = Path("skills/engineering_question_with_context.md")
if not composite_skill_path.parent.exists():
    composite_skill_path = Path("MP3/Part A/skills/engineering_question_with_context.md")
composite_skill_path.write_text(COMPOSITE_SKILL, encoding="utf-8")
print(f"Composite skill written to {composite_skill_path.resolve()}")

In [ ]:
# Step 1 — Print your composition skill so it's part of the cell output.
composite_skill_path = Path("skills/engineering_question_with_context.md")
if not composite_skill_path.exists():
    composite_skill_path = Path("MP3/Part A/skills/engineering_question_with_context.md")

print("=" * 62)
print(f"{composite_skill_path.name}")
print("=" * 62)
if composite_skill_path.exists():
    print(composite_skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

In [ ]:
# Step 2 — WITH skill: load skill as system prompt, expose both tools.

composite_query = (
    "I'm reviewing a bracket design for a project similar to your CardioSense work. "
    "The bracket needs to handle a 500 lbf load. What did Ridgeline use for the "
    "CardioSense enclosure, and what's that load in newtons?"
)

tools_both = [rag_tool_good, convert_units_tool]
tool_handlers_both = {
    "query_ridgeline_rag": query_ridgeline_rag,
    "convert_units":       convert_units,
}

result_with_skill = run_with_logging(
    messages=[{"role": "user", "content": composite_query}],
    tools=tools_both,
    tool_handlers=tool_handlers_both,
    system=composite_skill_path.read_text(encoding="utf-8"),
    label="section5_with_skill",
)

In [ ]:
# Step 3 — WITHOUT skill: same query, same tools, no system prompt.

result_without_skill = run_with_logging(
    messages=[{"role": "user", "content": composite_query}],
    tools=tools_both,
    tool_handlers=tool_handlers_both,
    label="section5_without_skill",
)

In [ ]:
# Step 4 — Summary table.

def tool_was_called(result, name):
    return any(t["name"] == name for t in result["tool_calls_made"])

def synthesis_present(result):
    """Heuristic: response mentions both newtons and CardioSense context."""
    txt = (result["final_message"] or "").lower()
    return "present" if ("newton" in txt or " n" in txt) and "cardiosense" in txt else "absent/partial"

for label, result in [("With skill", result_with_skill), ("Without skill", result_without_skill)]:
    rag_called = tool_was_called(result, "query_ridgeline_rag")
    conv_called = tool_was_called(result, "convert_units")
    synth = synthesis_present(result)
    print(f"{label:>14}:  rag_called = {str(rag_called):<5}  converter_called = {str(conv_called):<5}  synthesis = {synth}")

### ✏️ YOUR OBSERVATION

A skill adds deterministic orchestration: by explicitly telling the AI *when* to call each tool and *how* to synthesize the results, the skill turns a probabilistic decision into a reliable workflow — the AI consistently calls both the RAG and the converter when the composite skill is present, rather than occasionally skipping one. The limit of what skills can do is that they are still instructions to a language model: if the model fails to follow them (due to context window pressure, conflicting instructions, or ambiguous phrasing), the skill provides no hard guarantee — it is a strong nudge, not an enforced procedure. For truly safety-critical engineering workflows, skills must be paired with post-hoc validation of tool call logs, not treated as sufficient on their own.

> **Forward connection.** Part B asks you to do this on your own
> engineering work — write skills, expose MiniClaw RAG as an MCP
> tool, and run a documented design review. The composition you just
> built is the seed of that workflow at MiniClaw scale.

---
## Section 6: Reflections [8 pts]

### Reflection 1 — Where do tools end and skills begin?

A *tool* is for when you need to give the AI access to a capability it cannot compute itself — external data, real-time calculation, file I/O. For example, a finite element solver or a stress-concentration lookup table should be a tool: the AI cannot reproduce them from training, and the output must be exact. A *skill* is for when you need the AI to use the right tools in the right order for a recurring class of task — for example, a skill that says whenever reviewing a weld spec, first call the materials RAG, then verify units, then flag against the D1.1 standard encodes engineering judgment that otherwise lives only in a senior engineer's head. The distinction: tools give capability; skills give judgment about when and how to exercise that capability.

### Reflection 2 — What's the durable lesson?

The underlying idea is that AI systems become reliable in proportion to how well you define the interface between them and the rest of your workflow — what inputs they receive, what they are allowed to call, and what they are required to produce before passing output downstream. MCP, function calling, and GitHub Models are today's wire formats; next year they may be replaced, but the principle of *exposing capability through well-described contracts* and *encoding workflow knowledge as reusable artifacts* will remain. An engineer who understands how to write a tool schema and a skill document can adapt to whatever orchestration layer ships next year, because the design thinking behind both is stable even when the syntax changes.

### Reflection 3 — What worries you?

The failure mode I'd watch for is *skill atrophy in the engineering team* — over time, if the AI handles unit-checking, RAG querying, and design review consistently, engineers may lose the habit of doing these things manually and stop noticing when the AI gets them wrong. A misconfigured RAG tool that returns stale data, or a skill with an outdated unit-conversion table, would produce confident-sounding but incorrect answers, and the team would have reduced capacity to catch the errors because those checks are no longer part of their routine. The mitigation is to treat AI tool outputs as *reviewed artifacts*, not trusted outputs — every engineer should periodically run a sanity check on a tool result manually, the same way you calibrate an instrument rather than trusting it indefinitely.